### Source Material:
- Madi, Ramin. "NLP/Labs/Week_12/NMT." GitHub, https://github.com/raminmohammadi/NLP/blob/main/Labs/Week_12/NMT/Assignment.ipynb


# CoSQL Neural Machine Translation using Encoder-Decoder model

**Project Summary**
- This model reads english ('question' column) and SQL ('sql_query' column) tuples as inputs to train the model
- After training, the model will return SQL generated command outputs based off of english inputs.

**Data Description**
- Data Training and Test was created using the "" technique
- Test and Training Data Location:  

1. **Encoder (Gated Recurrent Unit (GRU)):**
   - The encoder processes the input (The English Phrased Questions) and converts it into a sequence of fixed-length numerical representations (embeddings).
   - It uses recurrent layer (GRU) 
   - The output is a context vector in sql.

2. **Decoder (TensorFlow):**
   - The decoder takes the context vector from the encoder and generates the SQL translation one statement at a time.
   - At each step, the decoder predicts the next statement in the sequence, based on the context vector and the previously generated statements.

3. **Attention Mechanism (Bahdanau):**
   - Attention allows the decoder to focus on relevant parts of the input sentence while translating, rather than relying solely on the fixed context vector. Useful for longer translations.

4. **Training (Loss Function = Adam):**
   - The model is trained on 660 English to SQL pairs.
   - Training Data can be found here: ../Data/Inputs/sql_training_full.csv
   - It minimizes a loss function (e.g., cross-entropy loss) to align the predicted English words with the actual translation.

5. **Inference:**
   - During translation, the decoder uses techniques like beam search to produce fluent and accurate SQL statements.

The encoder-decoder architecture enables the model to capture complex language relationships and produce high-quality translations.

In [24]:
# retreive most recent version of tensorflow
# %pip install --upgrade tensorflow
# retreive most recent version of tqdm
# %pip install --upgrade tqdm

**Imports and Libraries**

In [25]:
import tensorflow as tf
import pathlib
import re
import pandas as pd
import random
from helper import create_dataset, tokenize, preprocess_sentence
from tqdm import tqdm

## Load Initial Training/Test Data .CSV format with these column names ##
- column names: "template_id", "question", "sql_query"
- file should contain **8** matching **template_id** field values, 8 different ways to say the same **question** field in english and the same exact **sql_query** field for the 8 entries
- **training/test** data is located and should be named exactly this:  ../../Data/Inputs/sql_training_full.csv

In [26]:
# import training and test data from sql_training_full.csv

# load csv file into a pandas dataframe
sql_train_full = pd.read_csv('../../Data/Inputs/sql_training_full.csv')


**Set New_Training_Test_Files Flag**
- If new_training_test_files = true then new tab delimited 


In [27]:
# If training/test file creation flag is set to True, this will generate new training.txt and test.txt files from the sql_training_full.csv file. If set to False, it will use the existing training and test files. 
New_Training_Test_Files = False

**split into 75% training data / 25% test data**

In [28]:

# create training data by selecting 6 of the 8 rows with template_id. put the other 2 rows into test data using sql_train_full
# if New_Training_Test_Files is set to True, create new training and test data. If set to False, use the existing training and test files.
# if New_training_test_Files is set to True, loop through sql_train_full selecting 6 random rows for every template_id and put the other 2 rows into test data
train_data = pd.DataFrame()
test_data = pd.DataFrame()
if New_Training_Test_Files:
    for template_id in sql_train_full['template_id'].unique():
        # select 6 random rows for every template_id
        train_rows = sql_train_full[sql_train_full['template_id'] == template_id].sample(n=6, random_state=42)
        # append to train_data
        train_data = pd.concat([train_data, train_rows])
        # put the other 2 rows into test data
        test_rows = sql_train_full[sql_train_full['template_id'] == template_id].drop(train_rows.index)
        # append to test_data
        test_data = pd.concat([test_data, test_rows])
    # else use the existing training and test files
else:
    train_data = pd.read_csv('../../data/inputs/train.txt', sep='\t', header=None, names=['question', 'sql_query'])
    test_data = pd.read_csv('../../data/inputs/test.txt', sep='\t', header=None, names=['question', 'sql_query'])

In [29]:
# return the total number of rows in the training and test data
print(f'Total number of rows in training data: {len(train_data)}')
print(f'Total number of rows in test data: {len(test_data)}')

Total number of rows in training data: 660
Total number of rows in test data: 220


In [30]:
# print train_data column headers
print(f'Train data columns: {train_data.columns.tolist()}')

Train data columns: ['question', 'sql_query']


#### Create train.txt and test.txt
- train data is stored here:  ../../Data/Inputs/train.txt
- test data is stored here: ../../Data/Inputs/test.txt
- training data rows are shuffled so that training is more random between each run

In [31]:
# store train_data and test_data into tuples of (input, target) delimited by tab character
train_data_tuples = list(zip(train_data['question'], train_data['sql_query']))
test_data_tuples = list(zip(test_data['question'], test_data['sql_query']))
# create train.txt file with train_data_tuples tab delimited and shuffle the rows before writing to the file
if New_Training_Test_Files:
    random.shuffle(train_data_tuples)
    with open('../../Data/Inputs/train.txt', 'w', encoding='utf-8') as f:
        for input_text, target_text in train_data_tuples:
            f.write(f'{input_text}\t{target_text}\n')

    # create test.txt file with test_data_tuples tab delimited
    random.shuffle(test_data_tuples)
    test_data_tuples = list(zip(test_data['question'], test_data['sql_query']))
    with open('../../Data/Inputs/test.txt', 'w', encoding='utf-8') as f:
        for input_text, target_text in test_data_tuples:
            f.write(f'{input_text}\t{target_text}\n')


## Preprocessing and tokenization

Preprocess the sentences to lower the cases, trimming whitespace, removing non-alphabetic characters.

Wrap the sentence with <start> and <end> tokens for sequence modeling

The create dataset function loads the dataset, preprocesses each sentence pair, and returns separate lists for source and target languages.
Tokinzation function tokenizes a list of sentences and converts them into padded numerical sequences suitable for neural network inputs.

In [32]:
path_to_file = '../../data/inputs/train.txt'

random.shuffle(train_data_tuples)

# num_examples = 660  # Set the number of examples to load
# count number of rows in train.txt and store in num_examples
with open(path_to_file, 'r', encoding='utf-8') as f:
    num_examples = sum(1 for line in f)
source_lang, target_lang = create_dataset(path_to_file, num_examples)

# Tokenize the source and target languages
input_tensor, input_tokenizer = tokenize(source_lang)
target_tensor, target_tokenizer = tokenize(target_lang)

# Vocabulary sizes
input_vocab_size = len(input_tokenizer.word_index) + 1
target_vocab_size = len(target_tokenizer.word_index) + 1

## Encoder class (Gated Recurrent Unit (GRU))
It uses an embedding layer and GRU for encoding the input sentence into context vectors.

In [33]:
# Encoder
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.enc_units,
                               return_sequences=True,
                               return_state=True,
                               recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, *state = self.gru(x, initial_state=hidden)
        return output, state[0] # defect in original code, state is a list of length 1, so we need to extract the first element

    def initialize_hidden_state(self):
        return tf.zeros((self.batch_sz, self.enc_units))

## Bahdanau Attention (Bahdanau)

The below code defines the attention mechanism for the model. It calculates a context vector by weighting encoder outputs based on relevance to the decoder's current state.

In [34]:
# Attention Mechanism
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights


## Decoder class (Gated Recurrent Unit (GRU))

It uses the attention mechanism, GRU, and a dense layer to generate output sequences.

In [35]:
# Decoder with Teacher Forcing
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz # set the batch size
        self.dec_units = dec_units #Set the Decoder Units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim) #Initialize the Embedding Layer
        #Initialize the GRU Layer
        self.gru = tf.keras.layers.GRU(self.dec_units,
                               return_sequences=True,
                               return_state=True,
                               recurrent_initializer='glorot_uniform')

        self.fc = tf.keras.layers.Dense(vocab_size) #Initialize the Fully Connected (Dense) Layer
        self.attention = BahdanauAttention(self.dec_units) #Initialize the Attention Mechanism

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, *state = self.gru(x, initial_state=hidden)
        output = tf.reshape(output, (-1, output.shape[2]))
        state = state[0]  # defect in original code, state is a list of length 1, so we need to extract the first element
        x = self.fc(output)
        return x, state, attention_weights

## Training

set training parameters: batch size, embedding dimension, and GRU unit size.
Create a TensorFlow dataset, shuffling it, and batches for training.

In [36]:
# Training configuration
BUFFER_SIZE = len(input_tensor)
BATCH_SIZE = 20  # Set the Batch Size
steps_per_epoch = len(input_tensor) // BATCH_SIZE
embedding_dim = 128  # Set the Embedding Dimension
units = 256  # Set the Number of Units

# print buffer size, batch size, steps per epoch, embedding dimension, and number of units
print(f'Buffer Size: {BUFFER_SIZE}')
print(f'Batch Size: {BATCH_SIZE}')
print(f'Steps per Epoch: {steps_per_epoch}')
print(f'Embedding Dimension: {embedding_dim}')
print(f'Number of Units: {units}')

Buffer Size: 660
Batch Size: 20
Steps per Epoch: 33
Embedding Dimension: 128
Number of Units: 256


In [37]:
# Dataset
dataset = tf.data.Dataset.from_tensor_slices((input_tensor, target_tensor))
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

In [38]:
# Initialize encoder and decoder
encoder = Encoder(input_vocab_size, embedding_dim, units, BATCH_SIZE)
decoder = Decoder(target_vocab_size, embedding_dim, units, BATCH_SIZE)


In [39]:
# Optimizer and loss function
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')


In [40]:
def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

A single training step with teacher forcing to improve sequence generation.



In [41]:
tf.config.run_functions_eagerly(True)

In [42]:

# Training step with teacher forcing
@tf.function
def train_step(inp, targ, enc_hidden):
    # return the shape of the called function
    # print(f"train_step function called with input shape: {inp.shape}, target shape: {targ.shape}, encoder hidden state shape: {enc_hidden.shape}")
    loss = 0
    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(inp, enc_hidden)
        dec_hidden = enc_hidden
        dec_input = tf.expand_dims([target_tokenizer.word_index['<start>']] * BATCH_SIZE, 1)

        for t in range(1, targ.shape[1]):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += loss_function(targ[:, t], predictions)
            dec_input = tf.expand_dims(targ[:, t], 1)  # Teacher forcing

    batch_loss = loss / int(targ.shape[1])
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return batch_loss


In [43]:
# Training Loop with multiple epochs
EPOCHS = 10  # Assign number of epochs

for epoch in range(EPOCHS):
    enc_hidden = encoder.initialize_hidden_state()
    total_loss = 0

    for (batch, (inp, targ)) in tqdm(enumerate(dataset.take(steps_per_epoch)), total=steps_per_epoch, desc="Training", unit="batch"):
        batch_loss = train_step(inp, targ, enc_hidden)
        total_loss += batch_loss

    print(f"Epoch {epoch+1} Loss {total_loss / steps_per_epoch:.4f}")


Training: 100%|██████████| 33/33 [00:33<00:00,  1.02s/batch]


Epoch 1 Loss 1.8345


Training: 100%|██████████| 33/33 [00:32<00:00,  1.01batch/s]


Epoch 2 Loss 0.8200


Training: 100%|██████████| 33/33 [00:33<00:00,  1.01s/batch]


...

Epoch 8 Loss 0.1855


Training: 100%|██████████| 33/33 [00:36<00:00,  1.11s/batch]


Epoch 9 Loss 0.1751


Training: 100%|██████████| 33/33 [00:33<00:00,  1.00s/batch]

Epoch 10 Loss 0.1648


## Evaluation

In [44]:
def evaluate(sentence):
    # Preprocess the input sentence
    sentence = preprocess_sentence(sentence)
    inputs = [input_tokenizer.word_index.get(word, 0) for word in sentence.split()]
    inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs], maxlen=input_tensor.shape[1], padding='post')
    inputs = tf.convert_to_tensor(inputs)

    # Encode the input sentence
    hidden = [tf.zeros((1, units))]
    enc_out, enc_hidden = encoder(inputs, hidden)

    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([target_tokenizer.word_index['<start>']], 0)

    result = ''

    # Decode the output sequence
    for t in range(target_tensor.shape[1]):
        predictions, dec_hidden, attention_weights = decoder(dec_input, dec_hidden, enc_out)
        predicted_id = tf.argmax(predictions[0]).numpy()

        if target_tokenizer.index_word[predicted_id] == '<end>':
            break

        result += target_tokenizer.index_word[predicted_id] + ' '

        # Use the predicted word as the next input
        dec_input = tf.expand_dims([predicted_id], 0)

    return result.strip()

## Testing

In [45]:
with open('../../data/inputs/test.txt', 'r', encoding='utf-8') as f:
    test_pairs = [line.rstrip('\n').split('\t', 1) for line in f if line.strip()]

# overwrite the results file if it exists
results_file_path = pathlib.Path("../../data/results/cosql_nmt_gru_seq2seq_model1_results.txt")
if results_file_path.exists():
    results_file_path.unlink()

# To test all sentences in the test.txt file, loop through each source/target pair and evaluate the source sentence.
def translate_test_sentences():
    for idx, pair in enumerate(test_pairs, start=1):
        source_sentence, target_sentence = pair
        f.write(f"Translating sentence {idx}: {source_sentence}\n")
        f.write(f"Actual translation: {target_sentence}\n")
        predicted_translation = evaluate(source_sentence)
        f.write(f"Predicted translation: {evaluate(source_sentence)}\n")
        f.write("\n")
        print(f"Translating sentence {idx}: {source_sentence}")
        print(f"Actual translation: {target_sentence}")
        print(f"Predicted translation: {predicted_translation}")
        print()
        # print("\n" + "="*50 + "\n")

with open(results_file_path, 'w', encoding='utf-8') as f:
    translate_test_sentences()


Translating sentence 1: Who put up the biggest point total in one playoff game?
Actual translation: SELECT PLAYER_NAME FROM boxscores ORDER BY PTS DESC LIMIT 1;
Predicted translation: select player_name from boxscores where team_abbreviation = 'cle' order by pts desc limit 1;

Translating sentence 2: Name the player with the most points in any one playoff game.
Actual translation: SELECT PLAYER_NAME FROM boxscores ORDER BY PTS DESC LIMIT 1;
Predicted translation: select player_name from boxscores where team_abbreviation = 'cle' order by pts asc limit 1;

Translating sentence 3: Which Boston player came up shortest on the scoreboard in a playoff game?
Actual translation: SELECT PLAYER_NAME FROM boxscores WHERE TEAM_ABBREVIATION = 'BOS' ORDER BY PTS ASC LIMIT 1;
Predicted translation: select player_name from boxscores where team_abbreviation = 'cle' order by pts desc limit 1;

Translating sentence 4: Name the Boston player with the lowest point total in a playoff game.
Actual translation